In [143]:
import numpy as np
import math
from functools import reduce

I recalled a theorem from number theory called [Lucas' Theorem][1] having to do with this kind of situation, where basically ${m \choose n}$ is divisible by $p$ if and only if ${m_i \choose n_i} = 0$ (i.e., $n_i > m_i$ in the convention used here) for one of the $i$ (where $m_i, n_i$ are coefficeints of the base-$p$ representations of $m$ and $n$). So I used this to first find a few valid pairs $(m, n)$ such that the entry ${m \choose n}$ in Pascal's triangle is not divisible by $7$. I did this thinking of $m$ as the row number and seeing how many $n$'s were not divisible by $7$ by Lucas' Theorem.

From that brute-force pattern emerged--I noticed that for each row, they counted from $1$ to $7$ by $1$'s, then $2$ to $14$ by $2$'s, $3$ to $21$ by $3$'s, etc. And then at some point the count reset back to $2$ to $14$ by $2$'s, $4$ to $28$ by $4$'s, etc. The natural counting made me think it's interesting that it always goes from $1$ to $7$ (like the last digit counting from $0$ to $6$ before carrying over $1$ to the next column), but then multiplied by something. This gave me the idea that it has to do with the digits of the row number in base $7$. Then noticing how it kind of "wraps around" from bigger numbers, then back to smaller starting numbers, it had to do with multiplying the digits (since digits also get smaller as you carry to the next column). From there, you can pretty quickly play around and see the formula is $\prod_{i=1}^{k} (d_i + 1)$ for each digit $d_i$ in the base-$7$ representation of row $m$. $k = \lceil \log_{7}(m+1) \rceil$ is the number of digits in $m$'s base-$7$ representation which we will use going forward.

For a given row $m$, letting $m = \sum_{i=1}^{\lceil \log_{7}(m+1) \rceil} d_{i} 7^{i-1}$ where $d_i$ is the base-$7$ representation digit, the number of values not divisible by $7$ is $P(m) =  \prod_{i=1}^{\lceil \log_{7}(m+1) \rceil} (d_i + 1)$. We can quickly verify this for the first $7$ rows listed on the initial problem statement. The first $7$ rows each only have $1$ digit ($0$ through $6$) so they match the formula as $P(m) = m+1$ for $m = 0,1,2,3,4,5,6$. We proceed inductively.

Let $k$ be the number of digits in the base-$7$ representation of the row number.
$$
f(k) = \sum_{m=0}^{7^k - 1} \prod_{i=1}^{\lceil \log_{7}(m+1) \rceil} (d_{i} + 1) = \sum_{m=0}^{7^k - 1} P(m)
$$
be the total number of entries not divisibly by $7$ counted in all rows with $k$ or fewer digits (leading $0$'s are allowed). We can clearly see if $k = 1$, that sums the first $7$ rows from the initial problem statement which is $f(1) = \sum_{i=0}^{6} (i+1) = 28$.

Suppose you have $f(k)$ already. For a given $m$ with $k$ digits, $P(m) = \prod_{i=1}^{k} (d_i + 1)$. Imagine adding one more digit $d_{k+1}$ to the left of this (i.e., essentially adding $d_{k+1} 7^{k}$ to $m$). Then it's clear that 
$$
P(m + d_{k+1}{7}^k) = \prod_{i=1}^{k + 1} (d_i + 1) = (d_{k+1} + 1) \prod_{i=1}^{k} (d_i + 1) = (d_{k+1} + 1) P(m)
$$

Looping through the various possible digits for $d_{k+1} = 0,1,2,3,4,5,6$, we get that the new numbers added would be
$$
\sum_{d_{k+1}=0}^{6} P(m + d_{k+1}7^k) = P(m) \sum_{d_{k+1}=0}^{6}(d_{k+1} + 1) = P(m)(1 + 2 + 3 + 4 + 5 + 6 + 7) = 28 P(m).
$$

Since $m$ was chosen arbitrarily, this formula applies anytime we go from $k$ to $k+1$ digits. To show how this applies to $f$, we get
$$
f(k+1) = \sum_{m=0}^{7^{k+1} - 1} P(m) = \sum_{m=0}^{7^{k} - 1} 28P(m) = 28f(k)
$$

Initializing with $f(1) = 28$, we get $f(m) = 28^m$. Finally to set a proper row limit (not just a number of digits which limits us to $7^k$). Let $L$ be a row limit such that $7^k \leq L < 7^{k+1}$ and define it's base-$7$ representation as
$$
L = d_0 7^k + d_1 7^{k-1} + d_2 7^{k-2} + \dots + d_{k+1} 7^{1} + d_{k} 
$$
with $d_0$ > 0 being a requirement (since otherwise $7^k > L$). We want to find the total number of entries that are not divisible by $7$ when there are $< L$ rows (so rows $0$ to $L-1$, inclusive).

We can solve this by sequentially adding the different sections based on the digits. Adding each digit (from the left) sequentially corresponds to rows in that section--so section $0$ adds up all rows from $0$ to $d_0 7^k$, section $1$ adds all rows from $d_0 7^k$ to $d_0 7^k + d_1 7^{k-1}$, section $2$ adds all rows from $d_0 7^k + d_1 7^{k-1}$ to $d_0 7^k + d_1 7^{k-1} + d_2 7^{k-2}$, and so on. In general, (initializing $d_{-1} = 0$), let section $s$ is
$$
A_s = \{ m \in \mathbb{N} :  d_{s-1} 7^{k+1-s} \leq m < d_{s} 7^{k-s} \}
$$

For $A_0$, the right $k-1$ digits can be anything from $0$ to $6$, but the leftmost digit must be $< d_0$. So the number of possible leftmost digits is given by the triangular number $T_{d_0} = \sum_{j=0}^{d_0 - 1} d_i = \frac{d_0(d_0 + 1)}{2}$. The $k-1$ digits on the right can be anything so that just gives $f(k - 1)$. So the overall result is
$$
\frac{d_0 (d_0 + 1)}{2}f(k - 1).
$$

Suppose we are interested in section $A_s$, $s > 0$. Then the leftmost $s$ digits, $d_0, d_1, d_2, \dots, d_{s-1}$, are "fixed" as a lower bound for the remaining numbers in this set. Similar to the previous argument, there leftmost non-fixed digit $d_s$ can only have options from $< d_s$, so $T_{d_s} = \frac{d_s(d_s + 1)}{2}$, and the $k - s - 1$ digts to the right of $d_s$ can be anything, which gives $f(k - s - 1)$. However, to count for the fact that the leftmost $s+1$ digits are either fixed or constrained, we have to maintain a multiplier $M_s = M_{s-1} (d_{s-1} + 1)$ (where we initialize $M_{0} = 1$ since section $0$ had no fixed digits). So we get
$$
M_{s} \frac{d_s (d_s + 1)}{2} f(k-s-1)
$$

We can easily update these at each step.

[1]: https://en.wikipedia.org/wiki/Lucas%27s_theorem

In [196]:
L = 10**9
prime = 7
base_prime_rep = []
n = L
while n > 0:
    base_prime_rep.append(n % prime)
    n //= prime

base_prime_rep = base_prime_rep[::-1]

In [198]:
ms = [1]
result = 0
for i, d in enumerate(base_prime_rep):
    result += ms[-1] * (d*(d+1))//2 * 28**(len(base_prime_rep) - i - 1)
    ms[-1] *= d+1

result

2129970655314432

In [154]:
pascal_cache = [0]*10**9

for i in range(7):
    pascal_cache[i] = i+1

def pascal_not_by_7(n):
    if pascal_cache[n] == 0:
        pascal_cache[n] = (n % 7 + 1) * pascal_not_by_7(n // 7)
    
    return pascal_cache[n]

In [173]:
sum(pascal_not_by_7(i) for i in range(10**9)[::-1])

2129970655314432

In [174]:
def t(n):
    return n*(n+1)//2


n = 10**9
n2 = n
N = []
factor = 1
ans = 0
while n2 > 0:
    N = [n2 % 7] + N
    n2 = n2 // 7
print(N)
while len(N) > 0:
    ans += factor * t(N[0])*28**(len(N)-1)
    if N[0] == 7:
        factor *= 2
    else:
        factor *= (N[0] + 1)
    N = N[1:]
    print(N, ans, factor)
print(ans)

[3, 3, 5, 3, 1, 6, 0, 0, 6, 1, 6]
[3, 5, 3, 1, 6, 0, 0, 6, 1, 6] 1777180600172544 4
[5, 3, 1, 6, 0, 0, 6, 1, 6] 2031063543054336 16
[3, 1, 6, 0, 0, 6, 1, 6] 2121736022654976 96
[1, 6, 0, 0, 6, 1, 6] 2129507949477888 384
[6, 0, 0, 6, 1, 6] 2129692995354624 768
[0, 0, 6, 1, 6] 2129970564169728 5376
[0, 6, 1, 6] 2129970564169728 5376
[6, 1, 6] 2129970564169728 5376
[1, 6] 2129970652680192 37632
[6] 2129970653733888 75264
[] 2129970655314432 526848
2129970655314432


In [70]:
fact_cache = {0:1, 1:1, 2:2, 3:6, 4:24, 5:120}

def fact(n):
    if n not in fact_cache:
        fact_cache[n] = n*fact(n-1)

    return fact_cache[n]

In [ ]:
s

2361

In [71]:
choose_cache = {}
def choose(n,k):
    if n < k: return 0

    if (n,k) not in choose_cache:
        choose_cache[(n,k)] =  fact(n) // (fact(n-k)*fact(k))

    return choose_cache[(n,k)]

In [72]:
rebase_cache = {}
def rebase(n, p=7):
    if (n,p) not in rebase_cache:
        ntest = n
        ret = []

        if n < p:
            rebase_cache[(n,p)] = np.array([n])

        else:
            while ntest > 0:
                ret.append(ntest % p)
                ntest //= p
            
            rebase_cache[(n,p)] = np.array(ret)
    
    return rebase_cache[(n,p)]

In [150]:
all_entries = {}
pascals_not_divisble_by_7 = {0: 1}
# m is the row of Pascal's triangle
for m in range(7, 10**3, 7):
    mi = rebase(m)
    pascals_not_divisble_by_7[m] = 1

    # n = 0 or n = m, then the result is 1 which is not divisble by 7
    for n in range(1, m):
        ni = rebase(n)

        if np.all(mi[:len(ni)] >= ni):
            pascals_not_divisble_by_7[m] += 1

    pascals_not_divisble_by_7[m] += 1

In [151]:
pascals_not_divisble_by_7

{0: 1,
 7: 2,
 14: 3,
 21: 4,
 28: 5,
 35: 6,
 42: 7,
 49: 2,
 56: 4,
 63: 6,
 70: 8,
 77: 10,
 84: 12,
 91: 14,
 98: 3,
 105: 6,
 112: 9,
 119: 12,
 126: 15,
 133: 18,
 140: 21,
 147: 4,
 154: 8,
 161: 12,
 168: 16,
 175: 20,
 182: 24,
 189: 28,
 196: 5,
 203: 10,
 210: 15,
 217: 20,
 224: 25,
 231: 30,
 238: 35,
 245: 6,
 252: 12,
 259: 18,
 266: 24,
 273: 30,
 280: 36,
 287: 42,
 294: 7,
 301: 14,
 308: 21,
 315: 28,
 322: 35,
 329: 42,
 336: 49,
 343: 2,
 350: 4,
 357: 6,
 364: 8,
 371: 10,
 378: 12,
 385: 14,
 392: 4,
 399: 8,
 406: 12,
 413: 16,
 420: 20,
 427: 24,
 434: 28,
 441: 6,
 448: 12,
 455: 18,
 462: 24,
 469: 30,
 476: 36,
 483: 42,
 490: 8,
 497: 16,
 504: 24,
 511: 32,
 518: 40,
 525: 48,
 532: 56,
 539: 10,
 546: 20,
 553: 30,
 560: 40,
 567: 50,
 574: 60,
 581: 70,
 588: 12,
 595: 24,
 602: 36,
 609: 48,
 616: 60,
 623: 72,
 630: 84,
 637: 14,
 644: 28,
 651: 42,
 658: 56,
 665: 70,
 672: 84,
 679: 98,
 686: 3,
 693: 6,
 700: 9,
 707: 12,
 714: 15,
 721: 18,
 728: 2